<a href="https://colab.research.google.com/github/3OMDEH/Machine-Learning-Projects-Beginner-/blob/main/AirQuality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
RANDOM_STATE = 42

# **Reading the Data**

In [ ]:
# Dependencies, if needed: %pip install pandas numpy matplotlib scikit-learn
# No ucimlrepo installation is needed for the CSV workflow below.

In [ ]:
# Path Reading and Splitting into x and y
# All the values in the csv file are separated by ";", and decimals use a comma.
# Place AirQualityUCI.csv beside this notebook, or change DATA_PATH.
# Source: https://archive.ics.uci.edu/dataset/360/air+quality
DATA_PATH = Path('AirQualityUCI.csv')
if not DATA_PATH.exists() and Path('/content/AirQualityUCI.csv').exists():
    DATA_PATH = Path('/content/AirQualityUCI.csv')  # Existing Colab location
if not DATA_PATH.exists():
    raise FileNotFoundError('Download AirQualityUCI.csv from UCI and set DATA_PATH above.')
df = pd.read_csv(DATA_PATH, sep=';', decimal=',', na_values=[-200])
# -200 means missing in BOTH features and target. Never impute the target.
df = df.loc[:, ~df.columns.str.startswith('Unnamed:')].copy()
rows_before = len(df)
df = df.dropna(subset=['Date', 'Time', 'CO(GT)']).copy()
df['timestamp'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H.%M.%S', errors='raise')
# Remove exact duplicate records together, keeping features and labels aligned.
df = df.drop_duplicates().sort_values('timestamp').set_index('timestamp')
if df.index.has_duplicates:
    raise ValueError('Conflicting records share a timestamp; inspect before splitting.')
print(f'Removed {rows_before - len(df)} blank, unlabeled, or duplicate rows.')

# Estimate same-hour CO from sensors/weather, without other reference-analyzer targets.
# Ground-truth pollutant columns are excluded so deployment needs no reference analyzer.
feature_columns = ['PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
                   'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']
x = df[feature_columns].copy()
y = df['CO(GT)'].copy()
# Chronological 60/20/20 split: past training, later development, latest testing.
train_end, dev_end = int(len(df) * 0.6), int(len(df) * 0.8)
if min(train_end, dev_end - train_end, len(df) - dev_end) < 2:
    raise ValueError('Not enough labeled observations for three chronological splits.')
x_train, y_train = x.iloc[:train_end], y.iloc[:train_end]
x_dev, y_dev = x.iloc[train_end:dev_end], y.iloc[train_end:dev_end]
x_test, y_test = x.iloc[dev_end:], y.iloc[dev_end:]
assert x_train.index.max() < x_dev.index.min() < x_test.index.min()
for features, labels in [(x_train, y_train), (x_dev, y_dev), (x_test, y_test)]:
    assert features.index.equals(labels.index) and labels.notna().all()
print('Training / development / test:', len(x_train), len(x_dev), len(x_test))

In [ ]:
print(x_test.shape)
print(x_train.shape)

In [ ]:
x.info() # Full Data Info

In [ ]:
x.shape # Full Data Shape

In [ ]:
x_train.info() # Training Features Info

In [ ]:
x_train.head(5)

The original notebook reset shuffled indices. Unsorted indices are not a model error. We now retain the timestamp index so features and targets remain aligned and the split is easy to inspect.

In [ ]:
# No reset is needed: features and labels share the same timestamp index.
display(x_train.head())

In [ ]:
# Inspect development data here; reserve test evaluation for the final section.
x_dev.info()

In [ ]:
#@title 'drop = false' <---Test
# Let's see what happens if we set drop=False
# First, create a sample DataFrame with a custom index
sample_df = pd.DataFrame({'A': [1, 2, 3]}, index=['a', 'b', 'c'])
print("Original DataFrame with custom index:")
display(sample_df)

# Reset index with drop=False
sample_df_reset_false = sample_df.reset_index(drop=False)
print("DataFrame after reset_index(drop=False):")
display(sample_df_reset_false)

# Notice how the original index 'a', 'b', 'c' is now a column named 'index'

# Handle Missing Values
In the data documentation, `-200` represents a missing value. The CSV loader now converts it to `NaN` for all columns. Missing CO targets are removed before splitting, never filled. Feature imputation is learned only when a model pipeline is fitted.

In [ ]:
# Replacing all appearances of -200 with NaN is handled by read_csv(na_values=[-200]).
# Keeping this conversion in one place avoids inconsistent target handling.

In [ ]:
x_train.isnull().sum() # This line now shows the actual null values in the dataset after converting all the appearances of '-200' to a null value.

In [ ]:
# Use training data for missingness decisions.
x_train.isna().mean().sort_values(ascending=False)

In [ ]:
# Blank Date/Time rows and missing targets were removed together before splitting.
# Do not drop feature rows independently after creating y_train/y_dev/y_test.
assert x_train.index.equals(y_train.index)
assert x_dev.index.equals(y_dev.index)
assert x_test.index.equals(y_test.index)

In [ ]:
# Identifying the Imputer
# Median filling is a simple starting point; indicators flag absent sensor readings.
# No fit occurs here. Each model learns its own imputation values on training only.
preprocessing = make_pipeline(
    SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True),
    StandardScaler(),
)
# Empty CSV columns are removed during loading. NMHC(GT) and other reference
# analyzer measurements are excluded by the explicit feature list above.

In [ ]:
# Exact duplicate records were removed BEFORE splitting, together with their labels.
# Repeated sensor values at different hours can be legitimate observations.
print(f'Duplicate training timestamps: {x_train.index.duplicated().sum()}')

### Handle Missing Values in Target Variable
Targets were cleaned jointly with features before splitting. Imputing a missing target would teach the model invented answers. Retaining a common timestamp index avoids the earlier independent filtering/reset steps.

In [ ]:
# Verify feature/target alignment after cleaning.
for name, features, labels in [('train', x_train, y_train),
                               ('development', x_dev, y_dev),
                               ('test', x_test, y_test)]:
    assert features.index.equals(labels.index)
    assert labels.notna().all()
    print(f'{name}: {features.shape}, {labels.shape}')

# Feature Scaling
We use `StandardScaler` inside each model pipeline. Both the imputer and scaler fit only on the model's training rows. Predictions automatically reuse those learned transformations. There is no need to manually maintain scaled feature arrays.

In [ ]:
# Pipeline combines preprocessing with the estimator below.
# Keep x_train/x_dev/x_test raw; call fit/predict on the complete pipeline.

# **Model Training: Linear Regression**
Now that our data is prepared, let's train a `LinearRegression` model.

In [ ]:
# Initialize the Linear Regression model with its complete preprocessing.
model = make_pipeline(clone(preprocessing), LinearRegression())
# Train the model; missing values and scaling are handled automatically.
model.fit(x_train, y_train)
print('Linear Regression model trained successfully.')

In [ ]:
# Model comparison uses development data only.
candidates = {
    'Mean baseline': make_pipeline(clone(preprocessing), DummyRegressor(strategy='mean')),
    'Linear regression': model,
    'Ridge regression': make_pipeline(clone(preprocessing), Ridge(alpha=10.0)),
}
def regression_metrics(actual, predicted):
    return {'MAE': mean_absolute_error(actual, predicted),
            'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)}
rows = []
for name, candidate in candidates.items():
    candidate.fit(x_train, y_train)
    rows.append({'model': name, **regression_metrics(y_dev, candidate.predict(x_dev))})
comparison = pd.DataFrame(rows).sort_values('RMSE')
display(comparison)
# Lower MAE/RMSE is better; R2 may be negative on a shifted time period.
# MAE and RMSE use the CO target unit: mg/m^3.

In [ ]:
# Select using development RMSE and refit on training + development data.
# Run the test evaluation only after model choices are finished.
best_name = comparison.iloc[0]['model']
final_model = clone(candidates[best_name])
final_model.fit(pd.concat([x_train, x_dev]), pd.concat([y_train, y_dev]))
y_pred = final_model.predict(x_test)
print('Selected model:', best_name)
print('Final test metrics:', regression_metrics(y_test, y_pred))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, y_pred, alpha=0.6)
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(limits, limits, 'r--')  # Red dashed line for perfect prediction
axes[0].set(title='Actual vs. Predicted CO(GT)', xlabel='Actual CO (mg/m^3)',
            ylabel='Predicted CO (mg/m^3)')
axes[1].plot(y_test.index, y_test.to_numpy() - y_pred, linewidth=0.7)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set(title='Residuals over time', xlabel='Time', ylabel='Actual - predicted (mg/m^3)')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print('Negative predictions:', int((y_pred < 0).sum()))
# Do not silently clip negative predictions; inspect them and report any constraint explicitly.

## What changed and what to try next
- [UCI describes this as hourly time-series data with sensor/concept drift](https://archive.ics.uci.edu/dataset/360/air+quality). Chronological evaluation asks whether a model trained in the past works on later readings. The old random-split R2 is not directly comparable.
- This is same-hour sensor calibration, not future forecasting. A forecast requires a prediction horizon and only features available before that horizon.
- Other ground-truth pollutant readings were valid only if available during use. The default feature set now uses sensors and weather so it does not depend on a reference analyzer.
- For further tuning, use `TimeSeriesSplit` on training data with the complete pipeline. Consider a time gap if your deployment horizon or overlapping windows require it.
- Keep the latest test period untouched until choices are final. Scores describe hours with observed CO; missing target hours cannot be scored.
- Inspect residuals for drift and periods when many sensors are missing. A separate analysis of such periods can reveal weaknesses hidden by average errors.

## Original comments and learning notes
The original cells below are preserved as reference. Some approaches were corrected in the executable workflow above.

<details>
<summary>Original cell 5</summary>

```text
#@title .
#!pip install ucimlrepo
```

</details>

<details>
<summary>Original cell 6</summary>

```text
#@title Path Reading and Splitting into x and y
df = pd.read_csv("/content/AirQualityUCI.csv", sep=';', decimal=',')
# All the values in the csv file are separated by ";",  and the numbers are separated by a commma.

x = df.drop('CO(GT)', axis=1)
y = df['CO(GT)']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
```

</details>

<details>
<summary>Original cell 12</summary>

```text
The indexes are random, they are not arranged, we need to fix that
```

</details>

<details>
<summary>Original cell 13</summary>

```text
#@title Reset the index of x_train | x_test to a default integer index
x_train = x_train.reset_index(drop=True) # if we set the drop parameter to false, the current index of the dataframe will be converted into a new column
x_test = x_test.reset_index(drop=True) # if we set the drop parameter to false, the current index of the dataframe will be converted into a new column

# Display the first few rows to show the new index
display(x_train.head())
```

</details>

<details>
<summary>Original cell 16</summary>

```text

#**Handle Missing Values**
In the data documentation, `-200` represents a missing value. We will replace these with `np.nan` and then impute all `NaN` values.
```

</details>

<details>
<summary>Original cell 17</summary>

```text
#@title Replacing the all the appearances of "-200" with NaN
x_train = x_train.replace(to_replace=-200, value=np.nan)
x_test = x_test.replace(to_replace=-200, value=np.nan)
```

</details>

<details>
<summary>Original cell 20</summary>

```text
# Drop rows where 'Date' or 'Time' are NaN
print(f"Shape of x_train before dropping rows with missing Date/Time: {x_train.shape}")
x_train.dropna(subset=['Date', 'Time'], inplace=True)
print(f"Shape of x_train after dropping rows with missing Date/Time: {x_train.shape}")

print("\nNumber of NaN values after dropping rows with missing Date/Time:")
display(x_train.isnull().sum())


x_test = x_test.dropna(subset=['Date', 'Time'])
print(f"\nShape of x_test after dropping rows with missing Date/Time: {x_test.shape}")
display(x_test.isnull().sum())
```

</details>

<details>
<summary>Original cell 21</summary>

```text
from sklearn.impute import SimpleImputer

# Drop the 'Unnamed: 15', 'Unnamed: 16', and 'NMHC(GT)' columns as they are entirely empty or have too many missing values
# These columns cause an issue with SimpleImputer and also contain no useful data.
x_train = x_train.drop(['Unnamed: 15', 'Unnamed: 16', 'NMHC(GT)'], axis=1)
x_test = x_test.drop(['Unnamed: 15', 'Unnamed: 16', 'NMHC(GT)'], axis=1)

# Identifying the Imputer
imputer = SimpleImputer(strategy='mean')

# Identify numerical columns (after dropping the empty ones)
numerical_cols_train = x_train.select_dtypes(include=np.number).columns
numerical_cols_test = x_test.select_dtypes(include=np.number).columns

# Apply imputer to numerical columns for x_train
x_train[numerical_cols_train] = imputer.fit_transform(x_train[numerical_cols_train])

# Apply imputer x_test (using the imputer fitted on x_train to avoid data leakage)
x_test[numerical_cols_test] = imputer.transform(x_test[numerical_cols_test])

print("\nNumber of NaN values after imputation for x_train:")
display(x_train.isnull().sum())

print("\nx_train head after handling all missing values:")
display(x_train.head())

print("\nNumber of NaN values after imputation for x_test:")
display(x_test.isnull().sum())

print("\nx_test head after handling all missing values:")
display(x_test.head())
```

</details>

<details>
<summary>Original cell 23</summary>

```text
### Handle Missing Values in Target Variable
We need to process the `y_train` and `y_test` target variables to replace the `-200` values (missing data) with `NaN` and then drop the corresponding rows from both features and targets to maintain data integrity.
```

</details>

<details>
<summary>Original cell 24</summary>

```text
import numpy as np

# Before making any further changes, we need to ensure that the indices of
# y_train and y_test are aligned with x_train and x_test.
# x_train and x_test were already reset to a default integer index (0, 1, 2, ...)
# in cell `c1b12a24`. y_train and y_test, however, still retain their original
# indices from the initial train_test_split. This misalignment causes the KeyError
# when trying to drop rows using indices from y_train on x_train.

# Reset the indices of y_train and y_test to a default integer index
# This operation must occur before checking for NaNs in y_train/y_test
# if we intend to use positional drops or aligned operations.
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Now that indices are aligned, proceed with replacing -200 with NaN
y_train = y_train.replace(-200, np.nan)
y_test = y_test.replace(-200, np.nan)

print("NaN values in y_train before dropping:")
display(y_train.isnull().sum())
print("NaN values in y_test before dropping:")
display(y_test.isnull().sum())

# Create boolean masks for non-NaN values in y_train and y_test
# This is a more robust way to drop corresponding rows while maintaining alignment,
# especially after multiple index manipulations.
valid_train_mask = y_train.notna()
valid_test_mask = y_test.notna()

# Apply the masks to filter x_train, y_train, x_test, and y_test
x_train = x_train[valid_train_mask]
y_train = y_train[valid_train_mask]
x_test = x_test[valid_test_mask]
y_test = y_test[valid_test_mask]

print("\nNaN values in y_train after dropping:")
display(y_train.isnull().sum())
print("NaN values in y_test after dropping:")
display(y_test.isnull().sum())

print("\nShapes after cleaning target variables:")
print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}, y_test shape: {y_test.shape}")
```

</details>

<details>
<summary>Original cell 25</summary>

```text
# **Feature Scaling**
We'll use `StandardScaler` to scale our numerical features. It's important to fit the scaler only on the training data (`x_train`) to prevent data leakage and then transform both training and test sets.
```

</details>

<details>
<summary>Original cell 26</summary>

```text
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit on x_train and transform both x_train and x_test
x_train_scaled = scaler.fit_transform(x_train.select_dtypes(include=np.number))
x_test_scaled = scaler.transform(x_test.select_dtypes(include=np.number))

# Convert scaled arrays back to DataFrames, retaining column names
x_train_scaled_df = pd.DataFrame(x_train_scaled, columns=x_train.select_dtypes(include=np.number).columns)
x_test_scaled_df = pd.DataFrame(x_test_scaled, columns=x_test.select_dtypes(include=np.number).columns)

print("x_train_scaled head:")
display(x_train_scaled_df.head())
print("\nx_test_scaled head:")
display(x_test_scaled_df.head())
```

</details>

<details>
<summary>Original cell 28</summary>

```text
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize the Linear Regression model
model = LinearRegression()

# Train the model
model.fit(x_train_scaled, y_train)

print("Linear Regression model trained successfully.")
```

</details>

<details>
<summary>Original cell 30</summary>

```text
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--') # Red dashed line for perfect prediction
plt.title('Actual vs. Predicted CO(GT) Values')
plt.xlabel('Actual CO(GT)')
plt.ylabel('Predicted CO(GT)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()
```

</details>